# Chấm điểm động tác chống đẩy — Bước 7: so sánh các mô hình

Đầu vào:
- `pushup_dataset.npz` (bước 3): bộ Mendeley, 24 người, 3 góc quay.
- `qevd_dataset.npz` (bước 6): bộ QEVD, góc trước, nhãn chi tiết theo loại lỗi.

Mọi mô hình có cùng đầu ra 4 số cho mỗi rep: `[sai, nong, hong, sap]`
(`nong` = xuống không đủ sâu, `hong` = mông nhô/hông võng, `sap` = đổ sập/chân chạm sàn).
Vị trí tay không tính là lỗi.

| Mô hình | Đầu vào | Loại |
|---|---|---|
| LSTM | toạ độ 12 khớp theo từng khung | deep learning (baseline) |
| ST-GCN | đồ thị khung xương, gộp đặc trưng bằng trung bình | deep learning |
| ST-GCN (mean+max) | như trên, gộp bằng trung bình + cực đại | deep learning |
| 1D-CNN đặc trưng | góc khớp, độ cao vai... tính theo từng khung | deep learning |
| MLP thống kê | thống kê (min, max, trung bình, độ lệch) của các đặc trưng trên cả rep | deep learning |
| Gradient boosting | thống kê các đặc trưng trên cả rep | **không** phải deep learning, dùng làm mốc |

Hai thí nghiệm:
- **A. Trên QEVD** (chia theo người tập): mô hình nào nhận ra từng loại lỗi tốt nhất khi nhãn rõ ràng?
- **B. Trên Mendeley** (chia theo người tập): thêm dữ liệu QEVD vào tập train có giúp không?

**Cách train (giống nhau cho mọi mô hình deep learning).** Train trên toàn bộ các fold không phải fold test,
số epoch cố định, learning rate giảm dần theo cosine, **không dừng sớm**. Bản đầu của notebook này dừng sớm bằng một fold validation;
với dữ liệu nhỏ, cách đó lấy mất một phần tư dữ liệu train và chọn điểm dừng khá may rủi, trong khi gradient boosting được train trên toàn bộ,
nên so sánh không công bằng.

**Hãy bật GPU** (Settings → Accelerator). Với 3 seed và có GPU, ước lượng khoảng 25–30 phút.

## 1. Cấu hình

In [ ]:
MODELS = ["lstm", "stgcn", "stgcn_max", "featcnn", "statmlp"]   # bỏ bớt tên nếu muốn chạy nhanh hơn
SEEDS = [0, 1, 2]        # mỗi seed là một lần train độc lập; các bảng lấy trung bình và có thêm bảng độ lệch giữa các seed
EPOCHS = 40              # cố định cho mọi mô hình deep learning, không tinh chỉnh riêng cho từng mô hình

## 2. Đọc dữ liệu

In [ ]:
import os, glob, copy, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

def find_one(name):
    found = glob.glob(f"/kaggle/working/**/{name}", recursive=True) + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert found, f"Không thấy {name}. Hãy Add Input dataset chứa file này."
    return found[0]

MEND_PATH, QEVD_PATH = find_one("pushup_dataset.npz"), find_one("qevd_dataset.npz")
OUT_DIR = "/kaggle/working/so_sanh_mo_hinh"
os.makedirs(OUT_DIR, exist_ok=True)

mend = np.load(MEND_PATH)
mX, m_good_bad, mview, msubject, mfold = mend["X2d"], mend["y"], mend["view"], mend["subject"], mend["fold"]
EDGES = [tuple(e) for e in mend["edges"]]
VIEW_NAMES = ["front", "side", "diagonal"]
ERR_NAMES = ["sai", "nong", "hong", "sap"]

qevd = np.load(QEVD_PATH)
quse = qevd["use"]
qX, qY, qfold = qevd["X2d"][quse], qevd["Y"][quse], qevd["fold"][quse]
qview = np.zeros(len(qX), dtype=np.int64)            # QEVD chỉ có góc trước

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Mendeley:", mX.shape, "| QEVD:", qX.shape, "| thiết bị:", device)

## 3. Nhãn loại lỗi của Mendeley

Bảng này là kết quả tự xem 24 video bad (`kiem_tra_bad/nhan_loi_bad.csv`), đã bỏ lỗi về vị trí tay:
- Người chỉ có lỗi vị trí tay (11, 13, 14): rep bad của họ không còn là "sai" và bị loại khỏi train lẫn test.
- Người có video bad lẫn rep đúng (`NOT_ALL_BAD`): rep bad của họ có nhãn không chắc chắn, cũng bị loại.

Phần còn lại gọi là **nhãn sạch** (`mclean`), dùng để train và để đánh giá.

In [ ]:
ERR = {
    1: ["nong"], 2: ["hong"], 3: ["hong"], 4: ["nong"], 5: ["nong", "hong"], 6: ["hong"], 7: ["sap"], 8: ["hong", "nong"],
    9: ["nong", "hong"], 10: ["hong"], 11: [], 12: ["sap"], 13: [], 14: [], 15: ["hong", "sap"], 16: ["nong"],
    17: ["sap"], 18: ["hong"], 19: ["hong", "sap"], 20: ["nong"], 21: ["nong"], 22: ["nong"], 23: ["hong"], 24: ["nong"],
}
NOT_ALL_BAD = [1, 3, 11, 12, 13, 15, 17]

mY = np.zeros((len(mX), 4), dtype=np.float32)
mY[:, 0] = m_good_bad
for i in np.where(m_good_bad == 1)[0]:
    for e in ERR[int(msubject[i])]:
        mY[i, ERR_NAMES.index(e)] = 1
only_hand_error = np.array([len(ERR[int(s)]) == 0 for s in msubject])
mclean = (m_good_bad == 0) | (~only_hand_error & ~np.isin(msubject, NOT_ALL_BAD))

print("Mendeley nhãn sạch:", int(mclean.sum()), "rep | đúng:", int((mY[mclean, 0] == 0).sum()), "| sai:", int(mY[mclean, 0].sum()),
      "| nong/hong/sap:", mY[mclean, 1:].sum(0).astype(int).tolist())
print("QEVD             :", len(qX), "rep | đúng:", int((qY[:, 0] == 0).sum()), "| sai:", int(qY[:, 0].sum()),
      "| nong/hong/sap:", qY[:, 1:].sum(0).astype(int).tolist())

## 4. Tăng cường dữ liệu

Lật trái/phải, co giãn và xoay nhẹ, che ngẫu nhiên khớp. Chỉ áp dụng khi train, và **chỉ cho các mô hình nhận toạ độ thô**
(LSTM, ST-GCN). Hai mô hình dùng đặc trưng hình học không dùng tăng cường: khi một khớp bị che (toạ độ về 0), mọi góc khớp
liên quan thành vô nghĩa, và thí nghiệm cho thấy bước này làm chúng kém đi.

In [ ]:
SWAP_LR = [1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10]


def augment(x, rng):
    x = x.copy()
    if rng.random() < 0.5:
        x = x[:, SWAP_LR]
        x[..., 0] = -x[..., 0]
    scale = rng.uniform(0.9, 1.1)
    a = np.deg2rad(rng.uniform(-10, 10))
    rot = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]], dtype=np.float32)
    x[..., :2] = (x[..., :2] @ rot.T) * scale
    x[:, rng.random(x.shape[1]) < 0.1] = 0.0
    return x


USE_AUGMENT = {"lstm": True, "stgcn": True, "stgcn_max": True, "featcnn": False, "statmlp": False}


class RepDataset(torch.utils.data.Dataset):
    def __init__(self, X, V, Y, train):                 # train=True: có tăng cường dữ liệu
        self.X, self.V, self.Y, self.train = X, V, Y, train
        self.rng = np.random.default_rng(0)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = augment(self.X[i], self.rng) if self.train else self.X[i]
        return torch.from_numpy(x), int(self.V[i]), torch.from_numpy(self.Y[i])

## 5. Các mô hình deep learning

Cả bốn đều nhận `x` có shape `(batch, 32 khung, 12 khớp, 3)` và mã góc quay, trả về 4 logit.

**Điểm khác nhau chính:**
- **ST-GCN** gốc gộp đặc trưng bằng *trung bình* trên mọi khung và mọi khớp. Trung bình làm mờ thông tin
  "ở điểm thấp nhất của rep, người tập xuống tới đâu".
- **ST-GCN (mean+max)** ghép thêm giá trị *cực đại*, giữ lại được khoảnh khắc nổi bật nhất.
- **1D-CNN đặc trưng** không nhận toạ độ thô. Hàm `frame_features` tính 16 đại lượng hình học cho từng khung
  (góc khuỷu, góc thân, góc gối, góc nách hai bên; độ cao vai/hông/khuỷu; khoảng cách hai cổ tay, hai khuỷu, hai vai),
  cộng 12 kênh visibility. Sau đó là 3 lớp tích chập theo thời gian.
- **MLP thống kê** lấy min, max, trung bình, độ lệch chuẩn theo thời gian của 16 đại lượng đó (cùng loại thông tin
  mà gradient boosting dùng), ghép với mã góc quay rồi đưa qua mạng nơ-ron 3 lớp.

Các độ cao và khoảng cách **không** chia thêm cho độ rộng vai. Dữ liệu đã được chuẩn hoá theo kích thước cơ thể ở bước 3;
ở góc ngang hai vai chồng lên nhau nên độ rộng vai gần bằng 0, chia cho nó sẽ thổi phồng và làm nhiễu đặc trưng.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, n_out=4, hidden=64, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(12 * 3 + 3, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden * 2, n_out))

    def forward(self, x, view_id):
        b, t = x.shape[:2]
        v = nn.functional.one_hot(view_id, 3).float()[:, None].expand(b, t, 3)
        out, _ = self.lstm(torch.cat([x.reshape(b, t, -1), v], dim=-1))
        return self.head(out.mean(dim=1))


def normalized_adjacency(n, edges):
    A = np.eye(n, dtype=np.float32)
    for a, b in edges:
        A[a, b] = A[b, a] = 1
    d = A.sum(1)
    return torch.tensor(A / np.sqrt(d[:, None] * d[None, :]))


class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, A, stride=1, kernel_t=9, dropout=0.3):
        super().__init__()
        self.register_buffer("A", A)
        self.edge_importance = nn.Parameter(torch.ones_like(A))
        self.gcn = nn.Conv2d(c_in, c_out, 1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, (kernel_t, 1), (stride, 1), (kernel_t // 2, 0)),
            nn.BatchNorm2d(c_out), nn.Dropout(dropout))
        if c_in == c_out and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(nn.Conv2d(c_in, c_out, 1, (stride, 1)), nn.BatchNorm2d(c_out))

    def forward(self, x):                                   # (B, C, T, V)
        h = torch.einsum("bctv,vw->bctw", self.gcn(x), self.A * self.edge_importance)
        return torch.relu(self.tcn(h) + self.residual(x))


class STGCN(nn.Module):
    def __init__(self, n_out=4, pool="mean", channels=(32, 32, 64, 64), dropout=0.3):
        super().__init__()
        A = normalized_adjacency(12, EDGES)
        self.pool = pool
        self.input_bn = nn.BatchNorm1d(3 * 12)
        blocks, c = [], 3
        for i, c_out in enumerate(channels):
            blocks.append(STGCNBlock(c, c_out, A, stride=2 if i == 2 else 1, dropout=dropout))
            c = c_out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear((2 * c if pool == "mean+max" else c) + 3, n_out))

    def forward(self, x, view_id):                          # x: (B, T, V, C)
        b, t, v, c = x.shape
        h = self.input_bn(x.permute(0, 2, 3, 1).reshape(b, v * c, t)).reshape(b, v, c, t).permute(0, 2, 3, 1)
        h = self.blocks(h)
        pooled = [h.mean(dim=(2, 3))] + ([h.amax(dim=(2, 3))] if self.pool == "mean+max" else [])
        return self.head(torch.cat(pooled + [nn.functional.one_hot(view_id, 3).float()], dim=-1))


def torch_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = (ba * bc).sum(-1) / (ba.norm(dim=-1) * bc.norm(dim=-1) + 1e-6)
    return torch.acos(cos.clamp(-1 + 1e-6, 1 - 1e-6)) / np.pi           # về khoảng 0..1


def frame_features(x):
    # x: (B, T, 12, 3) -> (B, 28, T): 16 đặc trưng hình học + 12 kênh visibility cho từng khung
    P, V = x[..., :2], x[..., 2]
    f = []
    for s in (0, 1):                                                    # trái, phải
        sh, el, wr, hip, kn, an = [P[:, :, j + s] for j in (0, 2, 4, 6, 8, 10)]
        f += [torch_angle(sh, el, wr), torch_angle(sh, hip, an), torch_angle(hip, kn, an), torch_angle(el, sh, hip)]
    y = lambda joints: P[:, :, joints, 1].mean(2)
    sh_y, el_y, wr_y, hip_y = y([0, 1]), y([2, 3]), y([4, 5]), y([6, 7])
    f += [wr_y - sh_y, hip_y - sh_y, el_y - sh_y,
          sh_y - sh_y.mean(1, keepdim=True), hip_y - hip_y.mean(1, keepdim=True),
          (P[:, :, 4] - P[:, :, 5]).norm(dim=-1), (P[:, :, 2] - P[:, :, 3]).norm(dim=-1), (P[:, :, 0] - P[:, :, 1]).norm(dim=-1)]
    return torch.cat([torch.stack(f, dim=1), V.permute(0, 2, 1)], dim=1)


class FeatCNN(nn.Module):
    def __init__(self, n_out=4, c=64, dropout=0.3):
        super().__init__()
        self.input_bn = nn.BatchNorm1d(28)
        self.net = nn.Sequential(
            nn.Conv1d(28, c, 5, padding=2), nn.BatchNorm1d(c), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(c, c, 5, padding=2), nn.BatchNorm1d(c), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(c, c, 3, padding=1), nn.BatchNorm1d(c), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * c + 3, n_out))

    def forward(self, x, view_id):
        h = self.net(self.input_bn(frame_features(x)))
        return self.head(torch.cat([h.mean(2), h.amax(2), nn.functional.one_hot(view_id, 3).float()], dim=1))


def stat_features(x):
    # (B, 76): min, max, trung bình, độ lệch chuẩn theo thời gian của 16 đặc trưng hình học + visibility trung bình của 12 khớp
    f = frame_features(x)
    geo, vis = f[:, :16], f[:, 16:]
    return torch.cat([geo.mean(2), geo.amax(2), geo.amin(2), geo.std(2), vis.mean(2)], dim=1)


class StatMLP(nn.Module):
    def __init__(self, n_out=4, dropout=0.3):
        super().__init__()
        self.input_bn = nn.BatchNorm1d(76)
        self.net = nn.Sequential(nn.Linear(76 + 3, 128), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, n_out))

    def forward(self, x, view_id):
        return self.net(torch.cat([self.input_bn(stat_features(x)), nn.functional.one_hot(view_id, 3).float()], dim=1))


MAKE_MODEL = {
    "lstm": LSTMClassifier,
    "stgcn": lambda: STGCN(pool="mean"),
    "stgcn_max": lambda: STGCN(pool="mean+max"),
    "featcnn": FeatCNN,
    "statmlp": StatMLP,
}
MODEL_LABEL = {"lstm": "LSTM", "stgcn": "ST-GCN", "stgcn_max": "ST-GCN (mean+max)", "featcnn": "1D-CNN đặc trưng",
               "statmlp": "MLP thống kê", "gbm": "Gradient boosting"}

for name in MODELS:
    print(f"{MODEL_LABEL[name]:20s} số tham số: {sum(p.numel() for p in MAKE_MODEL[name]().parameters()):,}")

## 6. Mốc so sánh: gradient boosting trên đặc trưng của rep

Với mỗi rep, tính các thống kê (min, max, trung bình, độ lệch chuẩn) của góc khớp, biên độ lên/xuống của vai và hông,
khoảng cách hai tay..., rồi cho `HistGradientBoostingClassifier` phân loại. Không có mạng nơ-ron nào ở đây.
Nếu mô hình deep learning không vượt được mốc này thì chưa chứng minh được lợi ích của deep learning.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier


def np_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = (ba * bc).sum(-1) / (np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def rep_features(X, view):
    P, V = X[..., :2], X[..., 2]
    f = {}
    for s, n in [(0, "L"), (1, "R")]:
        sh, el, wr, hip, kn, an = [P[:, :, j + s] for j in (0, 2, 4, 6, 8, 10)]
        for name, a in [("elbow", np_angle(sh, el, wr)), ("body", np_angle(sh, hip, an)),
                        ("knee", np_angle(hip, kn, an)), ("armpit", np_angle(el, sh, hip))]:
            for stat, fn in [("min", np.min), ("max", np.max), ("mean", np.mean), ("std", np.std)]:
                f[f"{name}_{n}_{stat}"] = fn(a, axis=1)
    sh_w = np.linalg.norm(P[:, :, 0] - P[:, :, 1], axis=-1)
    sh_y, el_y, wr_y, hip_y = (P[:, :, j, 1].mean(2) for j in ([0, 1], [2, 3], [4, 5], [6, 7]))
    f["sh_range"] = np.ptp(sh_y, axis=1)
    f["hip_range"] = np.ptp(hip_y, axis=1)
    f["el_range"] = np.ptp(el_y, axis=1)
    f["sh_wr_min"] = (wr_y - sh_y).min(1)
    f["sh_wr_max"] = (wr_y - sh_y).max(1)
    f["sh_hip_mean"] = (hip_y - sh_y).mean(1)
    f["sh_hip_range"] = np.ptp(hip_y - sh_y, axis=1)
    f["wrist_w"] = np.linalg.norm(P[:, :, 4] - P[:, :, 5], axis=-1).mean(1)
    f["elbow_w_max"] = np.linalg.norm(P[:, :, 2] - P[:, :, 3], axis=-1).max(1)
    f["sh_w_mean"] = sh_w.mean(1)
    f["sh_w_range"] = np.ptp(sh_w, axis=1)
    for j in range(12):
        f[f"vis{j}"] = V[:, :, j].mean(1)
    for v in range(3):
        f[f"view{v}"] = (view == v).astype(np.float32)
    return pd.DataFrame(f)


def fit_predict_gbm(train, test):
    Ftr, Fte = rep_features(train[0], train[1]), rep_features(test[0], test[1])
    prob = np.zeros((len(Fte), 4), dtype=np.float32)
    for i in range(4):
        if 0 < train[2][:, i].mean() < 1:
            model = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05, random_state=0)
            prob[:, i] = model.fit(Ftr, train[2][:, i]).predict_proba(Fte)[:, 1]
    return prob

## 7. Hàm train

Giống nhau cho mọi mô hình deep learning: AdamW, binary cross-entropy trên cả 4 đầu ra, `EPOCHS` epoch,
learning rate giảm dần theo cosine, không dừng sớm.

In [ ]:
@torch.no_grad()
def predict(model, X, V):
    model.eval()
    loader = torch.utils.data.DataLoader(RepDataset(X, V, np.zeros((len(X), 4), np.float32), train=False), batch_size=256)
    return torch.cat([torch.sigmoid(model(xb.to(device), vb.to(device))).cpu() for xb, vb, _ in loader]).numpy()


def fit(name, train, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = MAKE_MODEL[name]().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    loader = torch.utils.data.DataLoader(RepDataset(*train, train=USE_AUGMENT[name]), batch_size=64, shuffle=True)
    for epoch in range(EPOCHS):
        model.train()
        for xb, vb, yb in loader:
            opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(model(xb.to(device), vb.to(device)), yb.to(device)).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()
    return model


def pick(X, V, Y, mask):
    return X[mask], V[mask], Y[mask]


def concat(a, b):
    return tuple(np.concatenate([x, y]) for x, y in zip(a, b))


def auc(y, p):
    return roc_auc_score(y, p) if 0 < y.mean() < 1 else np.nan


def binary_metrics(y, p, threshold=0.5):
    q = p > threshold
    tp, fp, fn, tn = (q & (y == 1)).sum(), (q & (y == 0)).sum(), (~q & (y == 1)).sum(), (~q & (y == 0)).sum()
    return dict(accuracy=(tp + tn) / len(y), recall=tp / max(tp + fn, 1), false_alarm=fp / max(fp + tn, 1))

## 8. Thí nghiệm A: trên QEVD, chia theo người tập

Fold `k` làm test, 4 fold còn lại để train. Mỗi rep được dự đoán đúng một lần bởi mô hình chưa từng thấy người tập đó.

In [ ]:
def run_qevd_cv(name, seed):
    prob = np.zeros((len(qX), 4), dtype=np.float32)
    for k in range(5):
        test = qfold == k
        if name == "gbm":
            prob[test] = fit_predict_gbm(pick(qX, qview, qY, ~test), pick(qX, qview, qY, test))
        else:
            model = fit(name, pick(qX, qview, qY, ~test), seed + k)
            prob[test] = predict(model, qX[test], qview[test])
    return prob


pred_A, rows = {}, []
for name in MODELS + ["gbm"]:
    for seed in (SEEDS if name != "gbm" else [0]):
        t0 = time.time()
        p = pred_A[(name, seed)] = run_qevd_cv(name, seed)
        row = {"mô hình": MODEL_LABEL[name], "seed": seed}
        row.update({f"AUC {e}": auc(qY[:, i], p[:, i]) for i, e in enumerate(ERR_NAMES)})
        row.update(binary_metrics(qY[:, 0], p[:, 0]))
        rows.append(row)
        print(f"[A] {MODEL_LABEL[name]:20s} seed {seed}: {time.time() - t0:.0f}s")

runs_A = pd.DataFrame(rows)
table_A = runs_A.groupby("mô hình", sort=False).mean(numeric_only=True).drop(columns="seed")
table_A.to_csv(os.path.join(OUT_DIR, "A_qevd.csv"))
print("Tỉ lệ rep sai trong QEVD:", round(float(qY[:, 0].mean()), 3), "(đoán 'sai' hết sẽ được accuracy bằng số này)")
display(table_A.round(3))
if len(SEEDS) > 1:
    print("Độ lệch chuẩn giữa các seed (gradient boosting chỉ chạy một lần nên để trống):")
    display(runs_A.groupby("mô hình", sort=False).std(numeric_only=True).drop(columns="seed")[[f"AUC {e}" for e in ERR_NAMES]].round(3))

**Cách đọc bảng A.** AUC = 0.5 là đoán bừa, 1.0 là hoàn hảo. Cột `AUC sai` là phân loại đúng/sai nói chung;
ba cột sau là từng loại lỗi. `accuracy`, `recall`, `false_alarm` tính cho đầu ra "sai" ở ngưỡng 0.5.
Lỗi `hong` được dự kiến là thấp ở mọi mô hình, vì QEVD quay từ phía trước nên hông bị thân che.

## 9. Thí nghiệm B: trên Mendeley, có và không có QEVD trong tập train

Chia 5 fold theo người tập như các bước trước. Chỉ dùng rep nhãn sạch để train và đánh giá.
- **Chỉ Mendeley**: train trên 4 fold Mendeley.
- **Mendeley + QEVD**: thêm toàn bộ rep QEVD vào tập train. Test vẫn là Mendeley.

In [ ]:
def run_mendeley_cv(name, seed, add_qevd):
    prob = np.zeros((len(mX), 4), dtype=np.float32)
    for k in range(5):
        test = mfold == k
        train = pick(mX, mview, mY, mclean & ~test)
        if add_qevd:
            train = concat(train, (qX, qview, qY))
        if name == "gbm":
            prob[test] = fit_predict_gbm(train, pick(mX, mview, mY, test))
        else:
            model = fit(name, train, seed + k)
            prob[test] = predict(model, mX[test], mview[test])
    return prob


good = mY[:, 0] == 0
pred_B, rows = {}, []
for name in MODELS + ["gbm"]:
    for add_qevd in (False, True):
        for seed in (SEEDS if name != "gbm" else [0]):
            t0 = time.time()
            p = pred_B[(name, add_qevd, seed)] = run_mendeley_cv(name, seed, add_qevd)
            row = {"mô hình": MODEL_LABEL[name], "train": "Mendeley + QEVD" if add_qevd else "chỉ Mendeley", "seed": seed}
            row["AUC sai"] = auc(mY[mclean, 0], p[mclean, 0])
            row.update(binary_metrics(mY[mclean, 0], p[mclean, 0]))
            for v, view_name in enumerate(VIEW_NAMES):
                mask = mclean & (mview == v)
                row[f"AUC sai {view_name}"] = auc(mY[mask, 0], p[mask, 0])
            for v, view_name in enumerate(VIEW_NAMES):      # rep nong so với rep đúng cùng góc quay
                mask = mclean & (mview == v) & (good | (mY[:, 1] == 1))
                row[f"AUC nong {view_name}"] = auc(mY[mask, 1], p[mask, 1])
            rows.append(row)
            print(f"[B] {MODEL_LABEL[name]:20s} {row['train']:16s} seed {seed}: {time.time() - t0:.0f}s")

runs_B = pd.DataFrame(rows)
table_B = runs_B.groupby(["mô hình", "train"], sort=False).mean(numeric_only=True).drop(columns="seed")
table_B.to_csv(os.path.join(OUT_DIR, "B_mendeley.csv"))
print("Tỉ lệ rep sai trong Mendeley nhãn sạch:", round(float(mY[mclean, 0].mean()), 3))
display(table_B.round(3))
if len(SEEDS) > 1:
    print("Độ lệch chuẩn giữa các seed:")
    display(runs_B.groupby(["mô hình", "train"], sort=False).std(numeric_only=True).drop(columns="seed")
            [["AUC sai", "AUC sai front", "AUC sai side", "AUC sai diagonal"]].round(3))

**Cách đọc bảng B.** Mỗi mô hình có hai dòng. So sánh hai dòng để biết QEVD có giúp không.
- `AUC sai`, `accuracy`, `recall`, `false_alarm`: phân loại đúng/sai trên toàn bộ Mendeley nhãn sạch.
- `AUC sai <góc>`: như trên, tách theo góc quay.
- `AUC nong <góc>`: mô hình phân biệt rep xuống nông với rep đúng tốt đến đâu, dùng đầu ra `nong`.

QEVD chỉ có góc trước, nên có thể giúp góc trước mà không giúp (hoặc làm kém) góc ngang.

## 10. Biểu đồ so sánh

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(21, 5))

table_A[[f"AUC {e}" for e in ERR_NAMES]].plot.bar(ax=axes[0], rot=20)
axes[0].set_title("A. QEVD: AUC theo loại lỗi"); axes[0].set_ylim(0.4, 1.0); axes[0].axhline(0.5, color="gray", ls="--")

for ax, col, title in [(axes[1], "AUC sai", "B. Mendeley: AUC đúng/sai"), (axes[2], "AUC nong front", "B. Mendeley góc trước: AUC lỗi nong")]:
    table_B[col].unstack("train").plot.bar(ax=ax, rot=20)
    ax.set_title(title); ax.set_ylim(0.4, 1.0); ax.axhline(0.5, color="gray", ls="--"); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "so_sanh.png"), dpi=120)
plt.show()

## 11. Mô hình deep learning nào tốt nhất?

Xếp hạng các mô hình deep learning theo từng tiêu chí (gradient boosting chỉ để tham chiếu).
Nếu một mô hình đứng đầu ở phần lớn tiêu chí thì chọn nó làm mô hình chính. Khi hai mô hình cách nhau ít hơn
độ lệch chuẩn giữa các seed (bảng ở mục 8 và 9) thì coi như ngang nhau.

In [ ]:
dl = [MODEL_LABEL[m] for m in MODELS]
both = table_B.xs("Mendeley + QEVD", level="train")
criteria = {
    "A: AUC sai (QEVD)": table_A["AUC sai"],
    "A: AUC nong (QEVD)": table_A["AUC nong"],
    "A: AUC sap (QEVD)": table_A["AUC sap"],
    "B: AUC sai (Mendeley + QEVD)": both["AUC sai"],
    "B: AUC nong góc trước": both["AUC nong front"],
    "B: AUC nong góc ngang": both["AUC nong side"],
}
summary = pd.DataFrame(criteria)
ranking = pd.DataFrame({k: v.loc[dl].idxmax() for k, v in criteria.items()}, index=["DL tốt nhất"]).T
ranking["điểm"] = [criteria[k].loc[w] for k, w in ranking["DL tốt nhất"].items()]
ranking["gradient boosting"] = [criteria[k].loc[MODEL_LABEL["gbm"]] for k in ranking.index]
display(summary.round(3))
display(ranking.round(3))
print("Số tiêu chí mỗi mô hình DL đứng đầu:", ranking["DL tốt nhất"].value_counts().to_dict())

## 12. Lưu dự đoán

In [ ]:
np.savez_compressed(
    os.path.join(OUT_DIR, "du_doan.npz"),
    **{f"A_{name}_s{seed}": p for (name, seed), p in pred_A.items()},
    **{f"B_{name}_{'qevd' if add else 'mend'}_s{seed}": p for (name, add, seed), p in pred_B.items()},
    qY=qY, qfold=qfold, mY=mY, mclean=mclean, mview=mview, msubject=msubject, mfold=mfold,
)
!ls -lh "{OUT_DIR}"